In [1]:
# Cell 1: Packing the Model for Hugging Face Hub Deployment

import os
import torch
import torch.nn as nn
from transformers import Wav2Vec2PreTrainedModel, Wav2Vec2Model, Wav2Vec2Config, AutoProcessor

# ==============================================================================
# Step 1: Define Core Pooling Component
# ==============================================================================
class MultiScaleAttentivePooling(torch.nn.Module):
    """
    Multi-Scale Attentive Pooling: Captures both weighted Mean and Std Deviation.
    Essental for detecting the 'unnatural smoothness' in AI-generated speech.
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, 128),
            torch.nn.Tanh(),
            torch.nn.Linear(128, 1)
        )

    def forward(self, x):
        # x shape: [batch, seq_len, 768]
        attn_weights = torch.softmax(self.attention(x), dim=1) 
        mu = torch.sum(attn_weights * x, dim=1) 
        delta = x - mu.unsqueeze(1) 
        var = torch.sum(attn_weights * (delta ** 2), dim=1)
        std = torch.sqrt(torch.clamp(var, min=1e-9)) 
        return torch.cat([mu, std], dim=-1) # Output dim: 1536

# ==============================================================================
# Step 2: Define Hugging Face Standard Wrapper Class
# ==============================================================================
class SentinelForDeepfakeDetection(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.wav2vec2 = Wav2Vec2Model(config)
        self.pooling = MultiScaleAttentivePooling(config.hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(config.hidden_size * 2, 512), 
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )
        self.post_init()

    def forward(self, input_values, attention_mask=None, labels=None):
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state
        pooled_output = self.pooling(last_hidden_state)
        logits = self.classifier(pooled_output)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)
        return {"loss": loss, "logits": logits} if loss is not None else logits

# ==============================================================================
# Step 3: Load .pth Weights and Convert to HF Standard
# ==============================================================================
checkpoint_path = "./model/best_deepfake_model.pth" 
export_dir = "./Sentinel_HF_Release"

print("🏗️ Initializing structure and injecting weights...")

if not os.path.exists(checkpoint_path):
    print(f"❌ Error: Checkpoint not found at {checkpoint_path}")
else:
    # 1. Initialize Configuration
    config = Wav2Vec2Config.from_pretrained("facebook/wav2vec2-base", num_labels=2)
    config.classifier_proj_size = 1536 
    
    # 2. Instantiate the Sentinel Model
    hf_model = SentinelForDeepfakeDetection(config)
    
    # 3. Weight Mapping (Remapping raw training keys to HF class structure)
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    new_state_dict = {}
    
    # Map 'backbone', 'pooling', and 'classifier' keys to their corresponding HF attributes
    for k, v in checkpoint['backbone'].items():
        new_state_dict[f"wav2vec2.{k}"] = v
    for k, v in checkpoint['pooling'].items():
        new_state_dict[f"pooling.{k}"] = v
    for k, v in checkpoint['classifier'].items():
        new_state_dict[f"classifier.{k}"] = v
        
    hf_model.load_state_dict(new_state_dict)
    
    # 4. Save the full deployment bundle
    hf_model.save_pretrained(export_dir)
    processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base")
    processor.save_pretrained(export_dir)
    
    print(f"\n✅ DONE! Bundle saved to: {export_dir}")

🏗️ Initializing structure and injecting weights...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ DONE! Bundle saved to: ./Sentinel_HF_Release


In [2]:
# Cell 2: Testing the Deployed Model on In_The_Wild Dataset

import os
import torch
import pandas as pd
import librosa
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoProcessor

# ==============================================================================
# 1. Environment & Model Preparation
# ==============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "./Sentinel_HF_Release"
data_dir = "./In_The_Wild"
meta_path = os.path.join(data_dir, "meta.csv")

print(f"🔍 Loading Sentinel Model from: {model_path}...")
# Loading via the custom wrapper class defined in Cell 1
# Ensure the 'SentinelForDeepfakeDetection' class remains in memory/namespace
model = SentinelForDeepfakeDetection.from_pretrained(model_path).to(device)
processor = AutoProcessor.from_pretrained(model_path)
model.eval()

# ==============================================================================
# 2. Loading Test Metadata
# ==============================================================================
# Mapping 'In_The_Wild' labels: 'spoof' -> 0 (Fake), 'bona-fide' -> 1 (Real)
# Aligns with our training logic: 0 is Fake, 1 is Real
df = pd.read_csv(meta_path)
label_map = {"spoof": 0, "bona-fide": 1}
df['target'] = df['label'].map(label_map)

print(f"📊 Testing on {len(df)} samples from In_The_Wild dataset...")

# ==============================================================================
# 3. Inference Loop
# ==============================================================================
predictions = []
true_labels = []

with torch.no_grad():
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Forensic Inference"):
        wav_path = os.path.join(data_dir, row['file'])
        
        try:
            # 1. Load Audio (Mandatory 16kHz for Wav2Vec2 backbone)
            speech, sr = librosa.load(wav_path, sr=16000)
            
            # 2. Pre-processing via Hugging Face Processor
            input_values = processor(speech, return_tensors="pt", sampling_rate=16000).input_values.to(device)
            
            # 3. Model Prediction (Logits output)
            logits = model(input_values)
            pred_idx = torch.argmax(logits, dim=1).item()
            
            predictions.append(pred_idx)
            true_labels.append(row['target'])
            
        except Exception as e:
            print(f"⚠️ Skipping {row['file']} due to loading error: {e}")

# ==============================================================================
# 4. Final Metrics & Report Generation
# ==============================================================================
acc = accuracy_score(true_labels, predictions)
print("\n" + "="*60)
print(f"🚀 FINAL TEST REPORT: Sentinel-Base vs In_The_Wild")
print(f"📍 Overall Accuracy: {acc:.4f}")
print("="*60)
# Detailed breakdown of Precision, Recall, and F1-Score
print(classification_report(
    true_labels, 
    predictions, 
    target_names=["Fake (Spoof)", "Real (Bona-fide)"]
))

🔍 Loading Sentinel Model from: ./Sentinel_HF_Release...


Loading weights:   0%|          | 0/226 [00:00<?, ?it/s]

📊 Testing on 31779 samples from In_The_Wild dataset...


Forensic Inference: 100%|██████████| 31779/31779 [04:12<00:00, 125.96it/s]


🚀 FINAL TEST REPORT: Sentinel-Base vs In_The_Wild
📍 Overall Accuracy: 0.9956
                  precision    recall  f1-score   support

    Fake (Spoof)       1.00      0.99      0.99     11816
Real (Bona-fide)       0.99      1.00      1.00     19963

        accuracy                           1.00     31779
       macro avg       1.00      0.99      1.00     31779
    weighted avg       1.00      1.00      1.00     31779



In [4]:
# Cell 3: Testing Remote Deployment on Hugging Face Hub

import os
import torch
import torch.nn as nn
import pandas as pd
import librosa
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
from transformers import Wav2Vec2PreTrainedModel, Wav2Vec2Model, AutoProcessor

# ==============================================================================
# 0. Core Architecture Definition (Required for Remote Loading)
# ==============================================================================
class MultiScaleAttentivePooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, 128),
            torch.nn.Tanh(),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        attn_weights = torch.softmax(self.attention(x), dim=1) 
        mu = torch.sum(attn_weights * x, dim=1) 
        delta = x - mu.unsqueeze(1) 
        var = torch.sum(attn_weights * (delta ** 2), dim=1)
        std = torch.sqrt(torch.clamp(var, min=1e-9)) 
        return torch.cat([mu, std], dim=-1)

class SentinelForDeepfakeDetection(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.wav2vec2 = Wav2Vec2Model(config)
        self.pooling = MultiScaleAttentivePooling(config.hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(config.hidden_size * 2, 512), 
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )
        self.post_init()

    def forward(self, input_values, attention_mask=None):
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        pooled_output = self.pooling(outputs.last_hidden_state)
        logits = self.classifier(pooled_output)
        return logits

# ==============================================================================
# 1. Remote Model & Data Preparation
# ==============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 🚀 Remote Repository
HF_REPO = "JesseHuang922/VoxSentinel-Base"
data_dir = "./In_The_Wild"
meta_path = os.path.join(data_dir, "meta.csv")

print(f"📡 Pulling Sentinel Model from Hugging Face: {HF_REPO}...")
model = SentinelForDeepfakeDetection.from_pretrained(HF_REPO).to(device)
processor = AutoProcessor.from_pretrained(HF_REPO)
model.eval()

# ==============================================================================
# 2. Loading Test Metadata with Robust Mapping
# ==============================================================================
if not os.path.exists(meta_path):
    raise FileNotFoundError(f"❌ Metadata file not found at: {meta_path}")

df = pd.read_csv(meta_path)

# 🛠️ Robust Label Mapping: Handles 'spoof', 'bona-fide', 'real', 'fake' case-insensitively
label_mapping = {
    "spoof": 0, "fake": 0, 
    "bona-fide": 1, "bonafide": 1, "real": 1
}

# Apply mapping and clean whitespace in filenames
df['target'] = df['label'].str.lower().str.strip().map(label_mapping)
df['file'] = df['file'].str.strip() 

# Safety check for unmapped labels
if df['target'].isnull().any():
    unknowns = df[df['target'].isnull()]['label'].unique()
    print(f"⚠️ Warning: Found unmapped labels: {unknowns}")
    df = df.dropna(subset=['target'])

print(f"📊 Testing on {len(df)} valid samples from In_The_Wild...")

# ==============================================================================
# 3. Remote Inference Loop
# ==============================================================================
predictions = []
true_labels = []

with torch.no_grad():
    for _, row in tqdm(df.iterrows(), total=len(df), desc="HF Remote Inference"):
        wav_path = os.path.join(data_dir, row['file'])
        
        try:
            # 1. Load & Resample
            speech, _ = librosa.load(wav_path, sr=16000)
            
            # 2. Preprocess (AutoProcessor uses Wav2Vec2FeatureExtractor internally)
            # return_attention_mask=True ensures stability for varying lengths if needed
            inputs = processor(speech, return_tensors="pt", sampling_rate=16000).input_values.to(device)
            
            # 3. Model Forward
            logits = model(inputs)
            pred_idx = torch.argmax(logits, dim=1).item()
            
            predictions.append(pred_idx)
            true_labels.append(int(row['target']))
            
        except Exception as e:
            print(f"⚠️ Skipping {row['file']} due to error: {e}")

# ==============================================================================
# 4. Global Performance Report
# ==============================================================================
if len(predictions) > 0:
    acc = accuracy_score(true_labels, predictions)
    print("\n" + "🌟"*25)
    print(f"🚀 REMOTE DEPLOYMENT TEST SUCCESSFUL")
    print(f"📍 Repo: {HF_REPO}")
    print(f"📍 Global Accuracy: {acc:.4f}")
    print("🌟"*25)
    
    # Detailed forensic metrics
    print(classification_report(true_labels, predictions, 
                                target_names=["Fake (0)", "Real (1)"], 
                                digits=4))
else:
    print("❌ No predictions were made. Check your data path and metadata.")

📡 Pulling Sentinel Model from Hugging Face: JesseHuang922/VoxSentinel-Base...


Loading weights:   0%|          | 0/226 [00:00<?, ?it/s]

📊 Testing on 31779 valid samples from In_The_Wild...


HF Remote Inference: 100%|██████████| 31779/31779 [03:49<00:00, 138.62it/s]


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
🚀 REMOTE DEPLOYMENT TEST SUCCESSFUL
📍 Repo: JesseHuang922/VoxSentinel-Base
📍 Global Accuracy: 0.9956
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
              precision    recall  f1-score   support

    Fake (0)     0.9987    0.9894    0.9940     11816
    Real (1)     0.9938    0.9992    0.9965     19963

    accuracy                         0.9956     31779
   macro avg     0.9962    0.9943    0.9953     31779
weighted avg     0.9956    0.9956    0.9956     31779

